# Network and Emotional Structure of Text as a Cognitive Signal

This notebook models short texts as networks instead of bags of words, then tests
whether the shape of that network — how connected it is, how many isolated concepts
it has, how emotionally expressive it is — carries a signal about the writer's
cognitive state.

The texts here are LLM-generated to simulate two writing styles: one resembling
Alzheimer's disease (AD) patients, one resembling age-matched healthy controls (HC).
The pipeline itself — turning text into a graph, testing which graph properties
differ between groups, then checking whether those properties can classify new
text — generalizes to any setting where you want to compare how two groups
structure their language: user research transcripts, support tickets, survey
open-ends, or written feedback grouped by segment.


In [ ]:
# One-time setup. EmoAtlas builds the text network and assigns NRC emotion
# scores; spaCy's large English model and NLTK's WordNet are its dependencies.
# !pip install git+https://github.com/MassimoStel/emoatlas
# !python -m spacy download en_core_web_lg
# import nltk; nltk.download('wordnet'); nltk.download('omw-1.4')


In [ ]:
import json

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

from emoatlas import EmoScores
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (roc_auc_score, accuracy_score, classification_report,
                              roc_curve, confusion_matrix)
from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.decomposition import PCA

emos = EmoScores(language="english")


## 1. Data

Each text is grouped by simulated condition (`alzheimer` / `healthy`) in a JSON file.
A small illustrative sample is included at `data/sample_merged_paragraphs.json` — short,
template-generated texts that mimic the format of the real corpus, used only to keep
this notebook runnable end to end. See `data/README.md` for how the real corpus was
generated and where to place it for a full run.


In [ ]:
with open("data/sample_merged_paragraphs.json") as f:
    data = json.load(f)

rows = []
for block in data:
    for text in block.get("alzheimer", []):
        rows.append({"text": text, "label": "alzheimer"})
    for text in block.get("healthy", []):
        rows.append({"text": text, "label": "healthy"})

df = pd.DataFrame(rows)
print(f"Loaded {len(df)} documents")
print(df["label"].value_counts())


## 2. From text to network: feature extraction

Each text is converted into a Textual Forma Mentis Network (TFMN) via EmoAtlas, which
links words based on syntactic dependency proximity rather than simple co-occurrence
counts. From the resulting graph we extract two feature families:

- **Topological features**: size and connectivity of the network (nodes, edges,
  density, clustering, number of components, largest-connected-component size,
  average path length, diameter, isolated nodes)
- **Emotional features**: z-scores for the 8 basic emotions in Plutchik's model,
  computed against the NRC Emotion Lexicon


In [ ]:
def extract_network_features(fmnt):
    G = nx.Graph()
    G.add_nodes_from(fmnt.vertices)
    G.add_edges_from(fmnt.edges)

    if G.number_of_nodes() == 0:
        return {}

    components = list(nx.connected_components(G))
    largest_component = G.subgraph(max(components, key=len)).copy()

    return {
        "n_nodes": G.number_of_nodes(),
        "n_edges": G.number_of_edges(),
        "density": nx.density(G),
        "mean_degree": np.mean([d for _, d in G.degree()]),
        "clustering": nx.average_clustering(G),
        "n_components": len(components),
        "lcc_size": len(largest_component),
        "lcc_ratio": len(largest_component) / G.number_of_nodes(),
        "avg_path_length": nx.average_shortest_path_length(largest_component)
                            if len(largest_component) > 1 else 0,
        "diameter": nx.diameter(largest_component) if len(largest_component) > 1 else 0,
        "n_isolated": len(list(nx.isolates(G))),
    }


def extract_features(text):
    fmnt = emos.formamentis_network(text)
    network_features = extract_network_features(fmnt)
    emotion_features = emos.emotions(text)
    return {**network_features, **emotion_features}


In [ ]:
records = []
for _, row in df.iterrows():
    features = extract_features(row["text"])
    features["label"] = row["label"]
    records.append(features)

df_features = pd.DataFrame(records).fillna(0)

TOPO_COLS = ["n_nodes", "n_edges", "density", "mean_degree", "clustering",
             "n_components", "lcc_size", "lcc_ratio", "avg_path_length",
             "diameter", "n_isolated"]
EMOTION_COLS = ["anger", "trust", "surprise", "disgust", "joy", "sadness", "fear", "anticipation"]

print(df_features.shape)
df_features.groupby("label")[TOPO_COLS].median().T.round(3)


## 3. Do the network structures actually differ?

A non-parametric Mann-Whitney U test compares each topological feature between
groups, with Benjamini-Hochberg FDR correction to control for testing 11 features
at once.


In [ ]:
alz = df_features[df_features["label"] == "alzheimer"]
hlt = df_features[df_features["label"] == "healthy"]

stat_results = []
for col in TOPO_COLS:
    stat, p = mannwhitneyu(alz[col], hlt[col], alternative="two-sided")
    effect_size_r = 1 - (2 * stat) / (len(alz) * len(hlt))
    stat_results.append({
        "feature": col,
        "alz_median": alz[col].median(),
        "hlt_median": hlt[col].median(),
        "p": p,
        "effect_size_r": round(effect_size_r, 3),
    })

stats_df = pd.DataFrame(stat_results)
_, stats_df["p_adj"], _, _ = multipletests(stats_df["p"], method="fdr_bh")
stats_df["significant"] = stats_df["p_adj"] < 0.05

stats_df.sort_values("p_adj").round(4)


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(TOPO_COLS):
    sns.boxplot(data=df_features, x="label", y=col, ax=axes[i],
                hue="label", palette={"alzheimer": "tomato", "healthy": "steelblue"}, legend=False)
    p_adj = stats_df.loc[stats_df["feature"] == col, "p_adj"].values[0]
    sig = "***" if p_adj < 0.001 else "**" if p_adj < 0.01 else "*" if p_adj < 0.05 else "ns"
    axes[i].set_title(f"{col}  {sig}", fontsize=10)
    axes[i].set_xlabel("")

for j in range(len(TOPO_COLS), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Topological features: Alzheimer-simulated vs. Healthy-simulated text", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


## 4. Emotional profile

Emotion z-scores against the NRC baseline, visualized per group.


In [ ]:
corpus_alz = " ".join(df.loc[df["label"] == "alzheimer", "text"])
corpus_healthy = " ".join(df.loc[df["label"] == "healthy", "text"])

emos.draw_statistically_significant_emotions(corpus_alz)


In [ ]:
emos.draw_statistically_significant_emotions(corpus_healthy)


## 5. Classification: can emotional profile alone separate the groups?

Emotion z-scores (8 features) are used to train two classifiers under 10-fold
stratified cross-validation, evaluated on accuracy and ROC AUC. Emotional features
were chosen over topological ones for this step specifically because they are
less tied to surface-level text length, which is a known confound with
LLM-generated synthetic text.


In [ ]:
X = df_features[EMOTION_COLS].values
y = LabelEncoder().fit_transform(df_features["label"])  # alzheimer=0, healthy=1

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "DecisionTree": DecisionTreeClassifier(max_depth=3, random_state=42),
}

classification_results = {}
for name, model in models.items():
    pipe = Pipeline([("scaler", StandardScaler()), ("clf", model)])
    y_prob = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba")[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    classification_results[name] = {"y_prob": y_prob, "y_pred": y_pred}
    print(f"{name}: AUC = {roc_auc_score(y, y_prob):.3f}, Accuracy = {accuracy_score(y, y_pred):.3f}")
    print(classification_report(y, y_pred, target_names=["Alzheimer", "Healthy"]))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for name, results in classification_results.items():
    fpr, tpr, _ = roc_curve(y, results["y_prob"])
    axes[0].plot(fpr, tpr, lw=2, label=f"{name} (AUC = {roc_auc_score(y, results['y_prob']):.3f})")

axes[0].plot([0, 1], [0, 1], "k--", lw=1)
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve")
axes[0].legend()

best_model = max(classification_results, key=lambda n: roc_auc_score(y, classification_results[n]["y_prob"]))
cm = confusion_matrix(y, classification_results[best_model]["y_pred"])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[1],
            xticklabels=["Alzheimer", "Healthy"], yticklabels=["Alzheimer", "Healthy"])
axes[1].set_title(f"Confusion Matrix ({best_model})")
axes[1].set_ylabel("True")
axes[1].set_xlabel("Predicted")

plt.tight_layout()
plt.show()


## 6. How separable are the groups, really?

If classification accuracy looks unusually high for a small, LLM-generated dataset,
it's worth checking whether that reflects a genuine cognitive-linguistic signal or
just low within-group variability (a known risk with synthetic text). LDA gives a
single axis of maximum separation; PCA checks whether that separation shows up
even without knowing the labels.


In [ ]:
X_scaled = StandardScaler().fit_transform(X)

lda = LinearDiscriminantAnalysis(n_components=1)
lda_scores = lda.fit_transform(X_scaled, y)

pca = PCA(n_components=2)
pca_scores = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for label_value, label_name, color in [(0, "Alzheimer", "tomato"), (1, "Healthy", "steelblue")]:
    mask = y == label_value
    axes[0].hist(lda_scores[mask, 0], bins=15, alpha=0.6, label=label_name, color=color)
    axes[1].scatter(pca_scores[mask, 0], pca_scores[mask, 1], alpha=0.6, label=label_name, color=color)

axes[0].set_xlabel("LDA Score")
axes[0].set_ylabel("Count")
axes[0].set_title("LDA Projection")
axes[0].legend()

axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
axes[1].set_title("2D PCA")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"PCA explained variance (PC1, PC2): {pca.explained_variance_ratio_[:2].round(3)}")


If both LDA and PCA show a clean, near-total separation between groups on
unlabeled structure alone, that's a flag, not a win. It suggests the two groups of
generated text may be too internally homogeneous and too different from each other
compared to what real patient and control language actually looks like. High
accuracy on this kind of data says more about the LLM's generation process than
about a validated clinical marker. See `docs/report.md` for the full discussion,
including why real transcript data (DementiaBank Pitt Corpus, ADReSS challenge)
is the natural next step before treating any of this as a clinical signal.
